# LA Studio Voice Design - VoxCPM2

This notebook loads exactly `voxcpm2` (`openbmb/VoxCPM2`) on CUDA.
It is independent from API Gateway and refuses every other model ID.

1. Choose **Runtime -> Change runtime type -> GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio's Voice Design panel.


In [ ]:
!nvidia-smi
!git clone --quiet https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM
!git -C /content/VoxCPM checkout --quiet 616d3d3e630a
%pip install -q -e /content/VoxCPM "soundfile==0.13.1" "fastapi==0.115.12" "uvicorn==0.34.3"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_voice_design_worker.py')
WORKER.write_text('import torch\n\nfrom voxcpm import VoxCPM\n\nMODEL_ID = "voxcpm2"\nMODEL_NAME = "VoxCPM2"\nUPSTREAM_MODEL = "openbmb/VoxCPM2"\nMODEL = VoxCPM.from_pretrained(UPSTREAM_MODEL, load_denoiser=False, optimize=True, device="cuda")\n\ndef design_with_exact_model(request):\n    instruction = ", ".join(part for part in (request.voice_description.strip(), request.style.strip()) if part)\n    instruction = re.sub(r"[()（）]", "", instruction).strip()\n    text = f"({instruction}){request.input}"\n    audio = MODEL.generate(\n        text=text,\n        cfg_value=2.0,\n        inference_timesteps=10,\n        seed=None if request.seed < 0 else request.seed,\n    )\n    return audio, int(MODEL.tts_model.sample_rate)\n\nimport io\nimport os\nimport re\nimport threading\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom fastapi import FastAPI, Header, HTTPException\nfrom fastapi.responses import Response\nfrom pydantic import BaseModel, Field\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nTOKEN = os.environ["LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN"]\nMAX_INPUT_CHARS = 4000\nMAX_OUTPUT_SECONDS = 300\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\n\nclass VoiceDesignRequest(BaseModel):\n    model: str = Field(min_length=1, max_length=120)\n    input: str = Field(min_length=1, max_length=MAX_INPUT_CHARS)\n    voice_description: str = Field(min_length=1, max_length=2000)\n    style: str = Field(default="", max_length=1000)\n    language: str = Field(default="en", max_length=40)\n    temperature: float = Field(default=0.9, ge=0.1, le=2.0)\n    seed: int = Field(default=-1, ge=-1)\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef audio_array(value):\n    if isinstance(value, (list, tuple)):\n        if not value:\n            raise RuntimeError("the selected model returned no audio")\n        value = value[0]\n    if torch.is_tensor(value):\n        value = value.detach().float().cpu().numpy()\n    audio = np.asarray(value, dtype=np.float32).reshape(-1)\n    if audio.size == 0 or not np.isfinite(audio).all():\n        raise RuntimeError("the selected model returned invalid audio")\n    return audio\n\ndef wav_response(value, sample_rate: int):\n    audio = audio_array(value)\n    if audio.size > int(sample_rate) * MAX_OUTPUT_SECONDS:\n        raise HTTPException(status_code=413, detail="generated audio exceeds the five minute output limit")\n    peak = float(np.max(np.abs(audio)))\n    if peak > 1.2:\n        audio = audio / peak\n    output = io.BytesIO()\n    sf.write(output, audio, int(sample_rate), format="WAV", subtype="PCM_16")\n    return Response(output.getvalue(), media_type="audio/wav", headers={"Cache-Control": "no-store"})\n\napp = FastAPI(title=f"LA Studio Voice Design - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "voice-design",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "formats": ["wav"],\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/voice_designs")\ndef voice_design(request: VoiceDesignRequest, authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    if request.model.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{request.model}\'. Open the notebook for the selected model.",\n        )\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab voice-design worker is busy; retry shortly")\n    try:\n        audio, sample_rate = design_with_exact_model(request)\n        return wav_response(audio, sample_rate)\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} voice design failed: {type(error).__name__}: {str(error)[:240]}",\n        ) from error\n    finally:\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
import os, re, secrets, subprocess, sys, time, urllib.request

MODEL_ID = 'voxcpm2'
TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env["LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN"] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "la_studio_voice_design_worker:app", "--host", "127.0.0.1", "--port", "3924"],
    cwd="/content",
    env=env,
)
for _ in range(180):
    try:
        check = urllib.request.Request(
            "http://127.0.0.1:3924/health",
            headers={"Authorization": "Bearer " + TOKEN},
        )
        with urllib.request.urlopen(check, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError("The exact-model worker did not become CUDA-ready. Inspect the cell output above.")

subprocess.run(
    ["bash", "-lc", "wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb"],
    check=True,
)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3924", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[^\s]+trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_VOICE_DESIGN_URL=" + public_url)
print("LA_STUDIO_COLAB_VOICE_DESIGN_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_VOICE_DESIGN_MODEL=" + MODEL_ID)
print("DEVICE=cuda; CPU_FALLBACK=false")
